### Data Ingestion

Without LangChain:

prompt -> llm answers

With LangChain:

- Upload PDF
- Convert into chunks
- Store in vector database
- Retrieve relevant parts
- Send to LLM
- Generate accurate answer

so basically it connects multiple steps together forming a pipeline acting like a chain

In [8]:
# document data structure
'''
Langchain document structure has two core components:
1. page_content(str) -> It contains all the details or contents of the pdf.
2. metadata(dict)-> It contains additional information related to the document such as source, total pages etc.
for eg:
    creating a document in langchain:
    doc = Document(
    page_content = "Rag is a technique....",
    metadata={
    "source" : "chapter1.pdf",
    "page" :5,
    "author":"Paulo",
    "timestamp" : "2024-02-03"
    }
    )
'''

from langchain_core.documents import Document

In [9]:
# creating a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

sample_texts = {
    "../data/text_files/python_intro.txt":"""Python Programming Introduction
    
Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular 
programming langues in the world.

Key features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.
""",
"../data/text_files/machine_learning.txt":"""Machine learning basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from
experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn
for themselves.

Types of machine learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
"""
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created.")

Sample text files created.


In [10]:
# To read files using TextLoader of Langchain
"""
A text loader:
Reads data from a source (file, URL, etc.)
Extracts the text content
Converts it into a format LangChain understands (called “documents”)
"""

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular \nprogramming langues in the world.\n\nKey features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.\n')]


In [11]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", #pattern to match files
    loader_cls=TextLoader, #loaders class to use, since this directory consists of text files so TextLoader class is used.
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
print(documents)

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from\nexperience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn\nfor themselves.\n\nTypes of machine learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n'), Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular \nprogramming langues in the world.\n\nKey features:\n- Easy to learn and use\n- Extensive sta

## RAG Pipelines - Data Ingestion to Vector DB 

### Complete flow: Load PDFs -> split into chunks -> create embeddings -> create vector store

In [ ]:
# 1. load all the pdf files

from langchain_community.document_loaders import PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader, # loader class for pdf files
    show_progress=False
)

pdf_documents = dir_loader.load()
print(pdf_documents)

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-05-31T14:05:48+05:45', 'source': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'file_path': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': '', 'author': 'Acer', 'subject': '', 'keywords': '', 'moddate': '2022-05-31T14:05:48+05:45', 'trapped': '', 'modDate': "D:20220531140548+05'45'", 'creationDate': "D:20220531140548+05'45'", 'page': 0}, page_content='Lect.Teksan \nUNIT-2: Intelligent Agents(4Hrs) \nSyllabus \n \n \n \nIntroduction of agents: \n \nSensor \n              For the detection of different actions and gestures of the surrounding a device or an \nelement is used, called sensor. After detection of any gesture, it provides the information to the \nelectronic device. A sensor is an electronic instrument that is able to measure the physical quantity \nand generate a considerate output. These outputs of the senso

In [18]:
#Breaks large PDFs into smaller chunks so they can 
# fit into LLM context windows and get better search results in your RAG system.


#2. Creating data chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    Parameters:
    - chunk_size: Maximum characters per chunk(adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks(preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, #each chunk ~1000 characters
        chunk_overlap = chunk_overlap, # 200 chars overlap for context
        length_function = len,
        separators=["\n\n", "\n", " ",""] #split hierarchy
    )
    #actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show what a chunk looks like
    if split_docs:
        print(f"Example chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks = split_documents(pdf_documents)
chunks

Split 29 documents into 82 chunks
Example chunk: 
Content: Lect.Teksan 
UNIT-2: Intelligent Agents(4Hrs) 
Syllabus 
 
 
 
Introduction of agents: 
 
Sensor 
              For the detection of different actions and gestures of the surrounding a device or an 
e...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-05-31T14:05:48+05:45', 'source': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'file_path': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': '', 'author': 'Acer', 'subject': '', 'keywords': '', 'moddate': '2022-05-31T14:05:48+05:45', 'trapped': '', 'modDate': "D:20220531140548+05'45'", 'creationDate': "D:20220531140548+05'45'", 'page': 0}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-05-31T14:05:48+05:45', 'source': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'file_path': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': '', 'author': 'Acer', 'subject': '', 'keywords': '', 'moddate': '2022-05-31T14:05:48+05:45', 'trapped': '', 'modDate': "D:20220531140548+05'45'", 'creationDate': "D:20220531140548+05'45'", 'page': 0}, page_content='Lect.Teksan \nUNIT-2: Intelligent Agents(4Hrs) \nSyllabus \n \n \n \nIntroduction of agents: \n \nSensor \n              For the detection of different actions and gestures of the surrounding a device or an \nelement is used, called sensor. After detection of any gesture, it provides the information to the \nelectronic device. A sensor is an electronic instrument that is able to measure the physical quantity \nand generate a considerate output. These outputs of the senso

### Embedding and VectorStoreDB

In [13]:
import numpy as np
#sentence_transformers converts the text into embeddings that is basically into vectors
from sentence_transformers import SentenceTransformer

#chromadb is a vectordb which stores vectors for similarity search
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    #the model here we are using is all-MiniLM-L6-v2 which is available in HuggingFace which is responsible for converting text into vectors
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    
    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_embedding_dimension()


#Initialize the embedding manager
embedding__manager = EmbeddingManager()
embedding__manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7570.42it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [27]:
class VectorStore:
    """Manages document embeddings in a chromadb vector store"""
    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the chromadb collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromadb client and collection"""
        try:
            # Creating persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing the vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")

        #preparing the data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generating unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #preparing metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())

        #Adding to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas = metadatas,
                documents= documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [20]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-05-31T14:05:48+05:45', 'source': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'file_path': '..\\data\\pdf\\AI Unit 2-Intelligent Agents.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': '', 'author': 'Acer', 'subject': '', 'keywords': '', 'moddate': '2022-05-31T14:05:48+05:45', 'trapped': '', 'modDate': "D:20220531140548+05'45'", 'creationDate': "D:20220531140548+05'45'", 'page': 0}, page_content='Lect.Teksan \nUNIT-2: Intelligent Agents(4Hrs) \nSyllabus \n \n \n \nIntroduction of agents: \n \nSensor \n              For the detection of different actions and gestures of the surrounding a device or an \nelement is used, called sensor. After detection of any gesture, it provides the information to the \nelectronic device. A sensor is an electronic instrument that is able to measure the physical quantity \nand generate a considerate output. These outputs of the senso

In [28]:
# 3. convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generating the embeddings
embeddings = embedding__manager.generate_embeddings(texts)


Generating embeddings for 82 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.13it/s]

Generated embeddings with shape: (82, 384)


In [29]:
# 4. Store in the vector database
vector_store.add_documents(chunks, embeddings)


Adding 82 documents to vector store...
Successfully added 82 documents to vector store
Total documents in collection: 82
